<a href="https://colab.research.google.com/github/eleni-z/ghflow-demo/blob/main/02_wed_attention_on_transformer_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Attention on Transformer Architecture
> Hands-on exploration of the OPT-125M model alongside Attention is All You Need.

Throughout the notebook, keep asking yourself concrete questions: What goes into this component? What does the component do? What comes out? What do the dimensions tell you about that calculation?

The goal is to help you see the actual operation of the transformer in yet another modality.

In [1]:
# Install necessary libraries
!pip install -q transformers accelerate

In [2]:
import torch
from transformers import GPT2Tokenizer, OPTForCausalLM

## Attention is All You Need: The Model Architecture - Section 3

The nomenclature of **encoder** and **decoder** is an old one, related to the behavior and purpose of the encoder and decoder.

<table width="100%" style="border-collapse: collapse; border: none; margin-bottom: 24px; background-color:white">
  <tr style="background-color:white">
    <td width="70%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-05a-model-definition.png?raw=1" alt="The paper's formal description of encoder and decoder" width="100%">
    </td>
    <td width="30%">
      <img src="https://media-cldnry.s-nbcnews.com/image/upload/t_fit-1000w,f_avif,q_auto:best/newscms/2017_05/1890591/170203-salt-bae-mn-1530.jpg" alt="A photo of Salt Bae sprinkling salt onto food" width="100%">
    </td>
  </tr>
</table>






Below, we see the big picture of the Transformer architecture in its encoder-decoder formulation for machine translation.


<table width="100%" style="border-collapse: collapse; border: none; margin-bottom: 24px; background-color:white">
  <tr style="background-color:white">
    <td width="35%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-04-transformer-architecture.png?raw=1" alt="Figure 1: The Transformer architecture" width="100%">
    </td>
    <td width="70%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-05b-encoder-decoder-stacks.png?raw=1" alt="The complete Section 3.1 encoder and decoder paragraphs" width="100%">
    </td>
  </tr>
</table>


## Generation with OPT

We first focus on the decoder-only architecture. The following process generates text using a transformer.

In [3]:
# Load the smallest OPT model and its corresponding tokenizer
# Note: OPT models use the GPT2Tokenizer class
model_name = "facebook/opt-125m"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = OPTForCausalLM.from_pretrained(model_name, attn_implementation="eager")

# Set the model to evaluation mode (no training)
model.eval()

# Define the input text
input_text = "Hello, I am a language model,"

# Tokenize the input text
inputs = tokenizer(input_text, return_tensors="pt")

# Generate tokens using the model
generate_ids = model.generate(inputs.input_ids, max_length=30)

# Decode and print the output
output_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print("Generated Output:")
print(output_text)

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  251MB            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  251MB            

model.safetensors: downloading bytes:           |  0.00B            

Generated Output:
Hello, I am a language model, and I am looking for a language that is easy to understand and that is easy to use. I am


However, this usage obscures the actual internal behavior of the transformer. In this exploration, we uncover, layer-by-layer, the internal operation of the transformer.

## 1. Model Architectural Choices

Run `model.config` below. These are the architectural decisions for the model we are using. We will return to them throughout the notebook because they determine the dimensions of the embedding tables, attention heads, decoder blocks, and MLP layers.

In [4]:
# look at configuration

model.config


OPTConfig {
  "_remove_final_layer_norm": false,
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "OPTForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "do_layer_norm_before": true,
  "dropout": 0.1,
  "dtype": "float16",
  "enable_bias": true,
  "eos_token_id": 2,
  "ffn_dim": 3072,
  "hidden_size": 768,
  "init_std": 0.02,
  "layer_norm_elementwise_affine": true,
  "layerdrop": 0.0,
  "max_position_embeddings": 2048,
  "model_type": "opt",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "prefix": "</s>",
  "tie_word_embeddings": true,
  "transformers_version": "5.21.0",
  "use_cache": true,
  "vocab_size": 50272,
  "word_embed_proj_dim": 768
}

### Ballpark Model Layers

The printed model is an indented outline. When one component is indented inside another, it belongs to that larger component. Use that nesting to find:

- the **transformer body**, `model.model`;
- the **decoder stack**, `model.model.decoder.layers`;
- one **decoder block**, such as `model.model.decoder.layers[0]`; and
- the **language modeling head**, `model.lm_head`, which sits outside the transformer body.

Within a decoder block, look for both self-attention and the MLP. A block is not the same thing as one attention head, and the 12 blocks in the stack are not the same as the 12 attention heads inside each block.

_One warning: the printed order tells you what modules are registered inside the model, but it does not always show the order in which activations move through them. In this OPT printout, for example, `final_layer_norm` appears before the decoder blocks even though the forward pass applies it after the blocks._

**Before you run:** On a piece of paper, sketch how you expect these parts to appear as an indented list. Then run the cell and compare your outline with the actual nesting.

In [5]:
# print model layers
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 768, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
      (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_norm): LayerNorm((768,), ep

## 2. From Text to Token Representations
### Tokenization

The tokenizer is used before the text enters the transformer. It turns the text into token IDs.

In [7]:
# Define the input text
input_text = "Hello, I am a language model,"

# Tokenize the input text
inputs = tokenizer(input_text, return_tensors="pt")

# Print total number of tokens:
print(f"> Total Number of Tokens: {len(inputs['input_ids'][0])}\n")
print(f"> Content of the 'inputs' variable:\n")

# Show actual outputs
inputs


> Total Number of Tokens: 9

> Content of the 'inputs' variable:



{'input_ids': tensor([[    2, 31414,     6,    38,   524,    10,  2777,  1421,     6]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

The text contains eight words or punctuation marks, but the tokenizer produced nine token IDs. **Why might the number of tokens differ from the number of words?** Give at least two reasons.

<details>
  <summary><font color="blue"><b>Check your reasoning</b></font></summary>
  <p>Two reasons are visible here. The tokenizer can divide text into subword tokens, so one word does not always correspond to one token. OPT also adds a special beginning-of-sequence token, which explains the extra token in this input.</p>
</details>

Now that we have token IDs, the next question is: **How does the model turn each ID into a learned vector representation?**

In [8]:
# See the tokens produced by the tokenizer
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

['</s>', 'Hello', ',', 'ĠI', 'Ġam', 'Ġa', 'Ġlanguage', 'Ġmodel', ',']

<br>

---

# The Transformer Body

## Token Representations

### The Token Embedding Matrix

Learned token embeddings basically operate as a lookup table. They are the initial learned representation (during training) of any individual token.

<font color='purple'>From the Vaswani paper:</font>

<img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-10a2-token-embeddings.png
?raw=1" alt="Sections 3.3 and 3.4" style="max-height:600px; width:auto; display:block; margin:auto;" width="80%">

<font color='purple'>What does this look like in practice?</font>

#### Structure

The token embedding matrix, $W_e$, transforms any token ID into its learned vector representation. An embedding layer does not multiply the token ID by the matrix. It uses the ID to look up one row, so the matrix acts as a learned lookup table.

In [12]:
# This is the token embedding weight matrix for OPT. Note that the actual output is truncated for visual reasons.
model.model.decoder.embed_tokens.weight

Parameter containing:
tensor([[ 0.1150, -0.1438,  0.0555,  ...,  0.2146,  0.0833,  0.0669],
        [ 0.1149, -0.1438,  0.0547,  ...,  0.2145,  0.0833,  0.0669],
        [ 0.0010, -0.0922,  0.1025,  ..., -0.0402,  0.0060, -0.1078],
        ...,
        [ 0.1152, -0.1437,  0.0547,  ...,  0.2145,  0.0833,  0.0671],
        [ 0.1151, -0.1455,  0.0546,  ...,  0.2156,  0.0837,  0.0673],
        [ 0.1156, -0.1437,  0.0577,  ...,  0.2139,  0.0833,  0.0650]],
       dtype=torch.float16, requires_grad=True)

In [13]:
# dimensions of the token embedding matrix/lookup table
model.model.decoder.embed_tokens.weight.shape

torch.Size([50272, 768])

#### Usage

Our input has nine token IDs, not every token in the vocabulary. The next cell uses those nine IDs to look up nine rows from the embedding matrix.

**Before you run:** How many rows should be returned? How long should each row be? If the input IDs changed, would the stored embedding matrix change, or would the model select different rows from the same matrix?

In [16]:
# Use the embedding lookup to transform the input IDs
token_embedding_activations = model.model.decoder.embed_tokens(inputs['input_ids'])
token_embedding_activations

tensor([[[ 0.0010, -0.0922,  0.1025,  ..., -0.0402,  0.0060, -0.1078],
         [ 0.0294, -0.1234,  0.0126,  ...,  0.1068,  0.0629, -0.0058],
         [ 0.0483, -0.0214, -0.0927,  ..., -0.0149, -0.0846,  0.0894],
         ...,
         [ 0.0062, -0.0082, -0.0017,  ...,  0.1227,  0.0453,  0.0600],
         [-0.0029,  0.0640,  0.0526,  ..., -0.0063, -0.0109, -0.0803],
         [ 0.0483, -0.0214, -0.0927,  ..., -0.0149, -0.0846,  0.0894]]],
       dtype=torch.float16, grad_fn=<EmbeddingBackward0>)

In [17]:
token_embedding_activations.shape

torch.Size([1, 9, 768])

### Weights and Activations

Compare the two tensors you just printed:

- `model.model.decoder.embed_tokens.weight` is the full learned lookup table stored in the model.
- `token_embedding_activations` is the output produced when that table is used for this particular sequence of nine token IDs.

The first tensor exists before a particular input is supplied as a part of the model. These are the **weights** that populate the layers of the model. The second exists because this input was processed, and is called **an activation**.

### Position Embeddings

Without recurrence or convolution, position must be supplied another way. The paper uses sinusoidal encodings; GPT-2 uses learned position embeddings.

<font color="purple">From the Vaswani paper:</font>

<img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-10b-positional-encoding.png?raw=1" alt="Complete Section 3.5" style="max-height:720px; width:auto; display:block; margin:auto;" width="70%">

<font color="purple">What does this look like in practice?</font>

#### Structure

Look back at `max_position_embeddings = 2048`. This is the largest number of token positions configured for this model, or more generally, the **context window**.

In [18]:
model.model.decoder.embed_positions.weight

Parameter containing:
tensor([[ 1.8272e-03,  9.0599e-04,  4.4289e-03,  ...,  1.6693e-02,
          1.7462e-03, -4.9057e-03],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [-3.5034e-02, -6.4648e-01,  3.7750e-02,  ..., -8.9783e-02,
          6.1493e-02, -1.0658e-02],
        ...,
        [-1.3721e-04,  2.5391e-02, -6.8703e-03,  ..., -4.1443e-02,
         -5.9776e-03,  2.0111e-02],
        [-2.1973e-03,  2.6764e-02, -3.1338e-03,  ..., -4.2297e-02,
          3.0947e-04,  2.0508e-02],
        [-4.5013e-03,  2.4933e-02, -3.7766e-03,  ..., -3.7994e-02,
         -1.3084e-03,  1.6785e-02]], dtype=torch.float16, requires_grad=True)

In [19]:
model.model.decoder.embed_positions.weight.shape

torch.Size([2050, 768])

#### Usage

Now use the position embedding matrix to produce activations for this particular input.

**Before you run:** What dimensions should the position embedding output activations have for a sequence of nine tokens?

In [21]:
position_embedding_activations = model.model.decoder.embed_positions(inputs['attention_mask'])
position_embedding_activations


tensor([[[-3.5034e-02, -6.4648e-01,  3.7750e-02,  ..., -8.9783e-02,
           6.1493e-02, -1.0658e-02],
         [ 6.0177e-04,  1.2573e-01, -2.2717e-03,  ..., -2.4277e-02,
          -3.8513e-02,  6.2103e-03],
         [ 3.0861e-03,  6.6467e-02, -7.9727e-03,  ..., -2.0828e-02,
          -2.2003e-02, -1.5465e-02],
         ...,
         [-9.8343e-03,  5.3589e-02, -4.1122e-03,  ..., -1.6708e-02,
          -1.1078e-02, -1.2032e-02],
         [-8.3771e-03,  4.9683e-02, -8.1787e-03,  ..., -1.7868e-02,
          -1.0086e-02, -1.1101e-02],
         [-5.6992e-03,  4.9469e-02, -9.2087e-03,  ..., -2.4551e-02,
          -3.5706e-03, -8.4839e-03]]], dtype=torch.float16,
       grad_fn=<EmbeddingBackward0>)

In [22]:
position_embedding_activations.shape

torch.Size([1, 9, 768])

### Combine Token and Position Information

For each token, we now have two vectors of the same length: a token embedding and a position embedding. The model adds the two vectors element by element.

**Before you run:** Predict the dimensions of the result. Are the values in the result stored model weights, or are they activations calculated for this input?

In [25]:
# Add token and position embedding activations
token_representations = token_embedding_activations + position_embedding_activations
token_representations


tensor([[[-3.4058e-02, -7.3877e-01,  1.4026e-01,  ..., -1.3000e-01,
           6.7444e-02, -1.1847e-01],
         [ 3.0029e-02,  2.3804e-03,  1.0361e-02,  ...,  8.2458e-02,
           2.4353e-02,  3.7384e-04],
         [ 5.1361e-02,  4.5044e-02, -1.0065e-01,  ..., -3.5675e-02,
          -1.0657e-01,  7.3914e-02],
         ...,
         [-3.6125e-03,  4.5441e-02, -5.8098e-03,  ...,  1.0596e-01,
           3.4241e-02,  4.7943e-02],
         [-1.1322e-02,  1.1365e-01,  4.4434e-02,  ..., -2.4124e-02,
          -2.1011e-02, -9.1431e-02],
         [ 4.2572e-02,  2.8030e-02, -1.0187e-01,  ..., -3.9398e-02,
          -8.8135e-02,  8.0872e-02]]], dtype=torch.float16,
       grad_fn=<AddBackward0>)

In [24]:
# Size
token_representations.shape

torch.Size([1, 9, 768])

---

## 3. One Decoder Layer

 <table width="100%" style="border-collapse: collapse; border: none; margin-bottom: 24px; background-color:white">
  <tr>
    <td width="70%">
    <p>
      One decoder block contains multi-head self-attention, <br> an MLP, layer norms, and residual connections. We can <br>see the two sublayers specified in the paper<br>as shown below.</p><br>
    </td>
    <td width="30%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-05c-masked-decoder-stack.png?raw=1" alt="Equation 1 and scaled dot-product attention diagram" style="max-height:600px; width:auto;">
    </td>
  </tr>
</table>

In [28]:
# look at decoder layers
model.model.decoder.layers[0]

OPTDecoderLayer(
  (self_attn): OPTAttention(
    (k_proj): Linear(in_features=768, out_features=768, bias=True)
    (v_proj): Linear(in_features=768, out_features=768, bias=True)
    (q_proj): Linear(in_features=768, out_features=768, bias=True)
    (out_proj): Linear(in_features=768, out_features=768, bias=True)
  )
  (activation_fn): ReLU()
  (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (fc1): Linear(in_features=768, out_features=3072, bias=True)
  (fc2): Linear(in_features=3072, out_features=768, bias=True)
  (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

### Layer Normalization Before Attention

This OPT model applies layer normalization before the attention calculation. The normalization changes the values, but it does not change the dimensions.

In [31]:
# apply layer 0 layer self-attention layer norm
ln_token_representations = model.model.decoder.layers[0].self_attn_layer_norm(token_representations)
ln_token_representations

tensor([[[-0.0594, -1.6621,  0.3423,  ..., -0.2783,  0.2167, -0.2242],
         [ 0.0608,  0.0170, -0.0205,  ...,  0.2681,  0.0628, -0.0891],
         [ 0.3804,  0.4182, -0.2212,  ...,  0.0111, -0.2391,  0.2893],
         ...,
         [-0.0530,  0.2388, -0.0511,  ...,  0.3940,  0.1416,  0.0848],
         [-0.0655,  0.5566,  0.1722,  ..., -0.1179, -0.0766, -0.3325],
         [ 0.3708,  0.3728, -0.2341,  ...,  0.0027, -0.1705,  0.3345]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

In [32]:
# look at shape
ln_token_representations.shape

torch.Size([1, 9, 768])

### Self-Attention

Attention allows every position directly examine every other relevant position using attention. That removes the sequential-computation bottleneck of RNNs and gives much shorter paths between distant pieces of information. Queries are search requests, other tokens have keys, reflecting the information they contain. The query is compared with keys to obtain weights to obtain relevance scores; after softmax, those scores are turned to weights, attention takes a weighted blend of the values.

<font color="purple">From the paper:</font>

<table width="100%" style="border-collapse: collapse; border: none; margin-bottom: 24px; background-color:white">
    <tr>
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-06-attention-definition-pt1.png?raw=1" alt="The paper's definition of attention" width="80%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-06-attention-definition-pt2.png?raw=1" alt="The paper's definition of attention" width="80%">
    </tr>
</table>

<font color="purple">What does this look like in practice?</font>

In [33]:
model.model.decoder.layers[0].self_attn

OPTAttention(
  (k_proj): Linear(in_features=768, out_features=768, bias=True)
  (v_proj): Linear(in_features=768, out_features=768, bias=True)
  (q_proj): Linear(in_features=768, out_features=768, bias=True)
  (out_proj): Linear(in_features=768, out_features=768, bias=True)
)

#### Multi-head Attention

<font color="purple">From the paper:</font>

<table width="100%" style="border-collapse: collapse; border: none; margin-bottom: 24px; background-color:white">
  <tr>
    <td width="70%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-08a-multi-head-text-pt1.png?raw=1" alt="Section 3.2.2 text" width="100%">
      <br></br>
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-08a-multi-head-text-pt2.png?raw=1" alt="Section 3.2.2b text" width="100%">
    </td>
    <td width="30%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-08b-multi-head-equations.png?raw=1" alt="Multi-head attention diagram and equations" width="100%">
    </td>
  </tr>
</table>

<font color="purple">What does this look like in practice?</font>

#### Step 1. Calculate Q, K, and V

The normalized token representations have dimensions `(1, 9, 768)`. Three different learned projections operate on the 768 features of every token. Each one produces a new `(1, 9, 768)` activation.

The dimensions match, but the three outputs have different jobs. Queries represent what a token position is looking for. Keys represent what each position can be matched on. Values contain the information that attention may combine.

We use the weight projection matrices $W_q$, $W_k$, and $W_v$ in order to transform the output of the embedding (or other representation) into the representation for each of the Q, K, and V heads.

<font color="purple"><b>Below, you will actually see the TRANSPOSED dimensions for these linear layers due to the way they are stored. They are backwards from what we see in the "module definitions". In other words, the dimensions you see are <code>(out_dimensions, in_dimensions)</code> rather than the (in_dimensions, out_dimensions) that we're used to seeing.</b></font>

In [34]:
# look at weight matrix for q - note that you can change the "q" for "k" and "v" to see others.
model.model.decoder.layers[0].self_attn.q_proj.weight

Parameter containing:
tensor([[ 0.0800,  0.2443, -0.2681,  ..., -0.2683, -0.0035, -0.0454],
        [-0.1362,  0.0869, -0.0869,  ...,  0.0484,  0.0303, -0.0471],
        [ 0.1505,  0.1498,  0.0703,  ...,  0.0029, -0.0348, -0.0657],
        ...,
        [ 0.0299,  0.1112,  0.0060,  ...,  0.0621, -0.0052,  0.0376],
        [ 0.0058,  0.0356,  0.0091,  ...,  0.0012,  0.0155,  0.0274],
        [-0.0443,  0.1431,  0.0268,  ..., -0.0435, -0.0213,  0.0091]],
       dtype=torch.float16, requires_grad=True)

In [35]:
# Print shapes of the matrices
print(f"Shape of W_q: {model.model.decoder.layers[0].self_attn.q_proj.weight.shape}")
print(f"Shape of W_k: {model.model.decoder.layers[0].self_attn.k_proj.weight.shape}")
print(f"Shape of W_v: {model.model.decoder.layers[0].self_attn.v_proj.weight.shape}")
print(f"Shape of W_out: {model.model.decoder.layers[0].self_attn.out_proj.weight.shape}")


Shape of W_q: torch.Size([768, 768])
Shape of W_k: torch.Size([768, 768])
Shape of W_v: torch.Size([768, 768])
Shape of W_out: torch.Size([768, 768])


Now, let's actually do the projection.

In [36]:
attention = model.model.decoder.layers[0].self_attn

queries = attention.q_proj(ln_token_representations)
keys = attention.k_proj(ln_token_representations)
values = attention.v_proj(ln_token_representations)

print("Queries:", queries.shape)
print("Keys:", keys.shape)
print("Values:", values.shape)

Queries: torch.Size([1, 9, 768])
Keys: torch.Size([1, 9, 768])
Values: torch.Size([1, 9, 768])


#### Step 2. Multi-head Attention |  Divide Q, K, and V Across the 12 Attention Heads

Each 768-feature projection contains 12 heads with 64 features per head. We separate those heads so that each head can calculate its own attention pattern. This reshape does not calculate new values. It only exposes the 12 groups already contained in the last dimension.

**Sometimes the way we conceptualize operations occuring versus how they are implemented differs.**

In [38]:
# Reshape to model heads
queries = queries.reshape(1, 9, 12, 64).transpose(1, 2)
keys = keys.reshape(1, 9, 12, 64).transpose(1, 2)
values = values.reshape(1, 9, 12, 64).transpose(1, 2)

print("Queries by head:", queries.shape)
print("Keys by head:", keys.shape)
print("Values by head:", values.shape)

Queries by head: torch.Size([1, 12, 9, 64])
Keys by head: torch.Size([1, 12, 9, 64])
Values by head: torch.Size([1, 12, 9, 64])


### Calculating Scaled Dot-Product (Masked) Attention

<font color="purple">From the paper:</font>

<table width="100%" style="border-collapse: collapse; border: none; margin-bottom: 24px; background-color:white">
  <tr style="background-color:white">
    <td width="70%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-07a-scaled-attention-text.png?raw=1" alt="Section 3.2.1 text" style="max-height:550px; width:auto;">
    </td>
    <td width="30%">
      <img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-07b-scaled-attention-equation.png?raw=1" alt="Equation 1 and scaled dot-product attention diagram" style="max-height:600px; width:auto;">
    </td>
  </tr>
</table>

<font color="purple">What does this look like in practice?</font>

#### Step 3. Calculate How Strongly Each Query Matches Each Key

For one head, each of the nine query vectors is multiplied by each of the nine key vectors. The result is a 9 by 9 score matrix. Each row identifies the token position asking the question. Each column identifies a position whose information might be used.

In [39]:
attention_scores = queries @ keys.transpose(-1, -2)
attention_scores = attention_scores / (64 ** 0.5)

print("Scaled attention scores:", attention_scores.shape)
# attention scores

attention_scores


Scaled attention scores: torch.Size([1, 12, 9, 9])


tensor([[[[-1.3740e+00, -1.0684e+00, -1.0713e+00, -1.1211e+00, -1.0967e+00,
           -9.5068e-01, -1.6787e+00, -2.3301e+00, -9.6338e-01],
          [ 1.8262e+00,  1.0693e+00,  8.1543e-01,  1.3096e+00,  1.4206e-02,
            9.0137e-01, -9.1455e-01,  9.7717e-02,  7.5146e-01],
          [ 1.2617e+00,  1.3428e+00,  5.8105e-01,  9.2969e-01,  1.0625e+00,
            8.1006e-01, -6.7969e-01,  2.9590e-01,  6.2988e-01],
          [ 9.6973e-01,  7.9248e-01,  6.4893e-01,  5.4541e-01,  2.0547e+00,
            9.6924e-01, -9.5312e-01,  9.4141e-01,  5.3467e-01],
          [ 9.4141e-01,  1.8750e-01,  6.9434e-01,  9.9414e-01, -3.8330e-01,
            9.9805e-01,  8.1055e-01, -4.0625e-01,  1.1133e-01],
          [ 6.1865e-01, -3.9429e-01, -1.2067e-01,  3.9404e-01,  4.0967e-01,
            6.3232e-01, -6.7627e-01, -9.1003e-02, -4.2114e-01],
          [-2.4573e-01, -1.8223e+00, -5.0293e-01, -4.5532e-01, -8.9258e-01,
            1.0291e-01, -1.0059e+00, -1.0039e+00, -4.9731e-01],
          [ 1.0273e+

#### Step 4. Apply the Causal Mask and Softmax

Rows in the mask represent query positions, and columns represent key and value positions. Values above the diagonal are negative infinity because those columns represent future positions. After softmax, those future positions will have probability 0.

In [41]:
causal_mask = torch.triu(
    torch.full(
        (1, 1, 9, 9),
        float("-inf"),
        device=attention_scores.device,
        dtype=attention_scores.dtype,
    ),
    diagonal=1,
)

In [42]:
masked_scores = attention_scores + causal_mask
attention_weights = torch.softmax(masked_scores, dim=-1)

print("Attention weights:", attention_weights.shape)
print("Row sums for head 0:", attention_weights[0, 0].sum(dim=-1))
attention_weights[0, 0]

Attention weights: torch.Size([1, 12, 9, 9])
Row sums for head 0: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=torch.float16,
       grad_fn=<SumBackward1>)


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6807, 0.3193, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3860, 0.4187, 0.1954, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3108, 0.2603, 0.2255, 0.2034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2800, 0.1317, 0.2188, 0.2952, 0.0745, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2240, 0.0814, 0.1069, 0.1790, 0.1818, 0.2271, 0.0000, 0.0000, 0.0000],
        [0.1924, 0.0398, 0.1487, 0.1560, 0.1007, 0.2725, 0.0900, 0.0000, 0.0000],
        [0.1107, 0.0817, 0.0768, 0.0968, 0.0180, 0.1070, 0.4377, 0.0714, 0.0000],
        [0.1575, 0.1455, 0.0706, 0.0806, 0.0880, 0.1187, 0.0330, 0.1181, 0.1880]],
       dtype=torch.float16, grad_fn=<SelectBackward0>)

#### Step 5. Use the Attention Weights to Combine the Value Vector

In [43]:
contextualized_values = attention_weights @ values
print("Weighted values by head:", contextualized_values.shape)

Weighted values by head: torch.Size([1, 12, 9, 64])


#### Step 6. Put the Head Outputs Back Together

Each token now has 12 results with 64 features each. Put those 12 groups back together so that every token has 768 features again. `out_proj` then applies a learned transformation to those 768 features for each token. It does not mix information between token positions.

In [44]:
combined_heads = contextualized_values.transpose(1, 2).reshape(1, 9, 768)
attention_output = attention.out_proj(combined_heads)

print("Combined heads:", combined_heads.shape)
print("Attention output:", attention_output.shape)

Combined heads: torch.Size([1, 9, 768])
Attention output: torch.Size([1, 9, 768])


### Add the Attention Residual

Attention produced a new `(1, 9, 768)` activation. The block adds it to the token representations that entered the block.

In [45]:
attention_sublayer_output = token_representations + attention_output
print("After the attention residual:", attention_sublayer_output.shape)

After the attention residual: torch.Size([1, 9, 768])


In [46]:
attention_sublayer_output.shape

torch.Size([1, 9, 768])

### The MLP Sublayer

- **MLP**: Multi-layer perceptron, also called
- **FFNN**: Feed-forward Neural Network

<font color="purple">From the paper: </font>

<img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-10a1-feed-forward.png?raw=1" alt="Sections 3.3 and 3.4" style="max-height:600px; width:auto; display:block; margin:auto;" width="80%">

<font color="purple">What does this look like in practice?</font>

In [47]:
print(f"MLP Layer Specification for fc1: {model.model.decoder.layers[0].fc1}")
print(f"MLP Layer Specification for fc2: {model.model.decoder.layers[0].fc2}\n")

# you can look at the weights yourself if you like. Change fc1 to fc2 as desired.
model.model.decoder.layers[0].fc1.weight

MLP Layer Specification for fc1: Linear(in_features=768, out_features=3072, bias=True)
MLP Layer Specification for fc2: Linear(in_features=3072, out_features=768, bias=True)



Parameter containing:
tensor([[-0.0871,  0.0035, -0.0213,  ...,  0.0425, -0.0021,  0.0082],
        [ 0.0037,  0.0230,  0.0150,  ...,  0.0206,  0.0172,  0.0158],
        [ 0.0125, -0.0129,  0.0245,  ...,  0.0040, -0.0299,  0.0462],
        ...,
        [ 0.0075,  0.0394,  0.0232,  ..., -0.0341,  0.0475,  0.0082],
        [-0.0055, -0.0038, -0.0399,  ...,  0.0158,  0.0176, -0.0500],
        [-0.0155,  0.0334,  0.0111,  ...,  0.0387, -0.0299,  0.0230]],
       dtype=torch.float16, requires_grad=True)

In [48]:
# apply the FFNN pre-layer norm
mlp_input = model.model.decoder.layers[0].final_layer_norm(attention_sublayer_output)
mlp_input.shape

torch.Size([1, 9, 768])

In [49]:
# use the first dimension of the FFNN to transform the attention sublayer output
fc1_output = model.model.decoder.layers[0].fc1(mlp_input)
fc1_output.shape

torch.Size([1, 9, 3072])

In [50]:
# apply the activation function for the FFNN
relu_output = model.model.decoder.layers[0].activation_fn(fc1_output)
relu_output.shape

torch.Size([1, 9, 3072])

In [51]:
# Project 3072 features back to the model's 768-feature residual width.
fc2_out = model.model.decoder.layers[0].fc2(relu_output)
fc2_out.shape

torch.Size([1, 9, 768])

In [52]:
# Add the residual the processed blow
mlp_output = attention_sublayer_output + fc2_out
print('Complete decoder block 0 output:', tuple(mlp_output.shape))

Complete decoder block 0 output: (1, 9, 768)


<br>

---

## 4. The Decoder Stack: Iteration and The Rest of the Transformer Body

We have finished block 0. Blocks 1 through 11 still need to operate on the activations, in order. Each block receives the output from the block before it. The dimensions usually remain `(1, 9, 768)`, but the values continue to change.

**Before you run:** The 12 decoder blocks and 12 attention heads describe two different kinds of repetition in this model. Explain.

In [53]:
# Iterate over the rest of the decoder layers/blocks
for layer_idx, layer in enumerate(model.model.decoder.layers):

  if layer_idx == 0:
    # skip it because we just did it above
    continue

  # otherwise, let's treat each decoder block as the operator and operate on each layer
  mlp_output = layer(hidden_states = mlp_output, attention_mask=causal_mask)
  print(f"Finished Layer {layer_idx}: Shape of output of MLP: {mlp_output.shape}")

Finished Layer 1: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 2: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 3: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 4: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 5: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 6: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 7: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 8: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 9: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 10: Shape of output of MLP: torch.Size([1, 9, 768])
Finished Layer 11: Shape of output of MLP: torch.Size([1, 9, 768])


### Final Layer Normalization

After the activations have passed through all 12 decoder blocks, the transformer body applies one final layer normalization. The language modeling head comes after the body.

In [54]:
final_hidden_state = model.model.decoder.final_layer_norm(mlp_output)
final_hidden_state.shape

torch.Size([1, 9, 768])

<br>

---

## 5. The Language Modeling Head

The transformer body has produced a 768-feature vector for each of the nine token positions. The language modeling head uses each of these 9 representations to calculate one logit (score) for every token in the vocabulary.


In [55]:
# The language modeling head dimensions
model.lm_head

Linear(in_features=768, out_features=50272, bias=False)

In [56]:
# operate on the hidden states using the LM head
logit_outputs = model.lm_head(final_hidden_state)
logit_outputs

tensor([[[-3.4551, -3.4531, 13.1562,  ..., -3.5098, -3.5254, -3.7480],
         [-9.9375, -9.9297, -1.0254,  ..., -9.9609, -9.8438, -9.7344],
         [-9.4453, -9.4375, -0.2155,  ..., -9.5078, -9.4688, -9.3203],
         ...,
         [-5.6445, -5.6406, -0.1624,  ..., -5.5703, -5.6328, -5.7344],
         [-6.6094, -6.5977,  0.9570,  ..., -6.5508, -6.5117, -6.7695],
         [-7.7578, -7.7578, -1.4268,  ..., -7.7266, -7.6992, -7.7148]]],
       dtype=torch.float16, grad_fn=<UnsafeViewBackward0>)

In [57]:
# look at the shape
logit_outputs.shape

torch.Size([1, 9, 50272])

### Weight Tying: The token embedding matrix and the LM Head

<font color='purple'>The token embedding matrix/table shows up again...</font>

<img src="https://github.com/csbell-vu/attention-exploration/blob/main/assets/paper-10a2-token-embeddings.png
?raw=1" alt="Sections 3.3 and 3.4" style="max-height:600px; width:auto; display:block; margin:auto;" width="80%">

<font color='purple'>What does this look like in practice?</font>

In this OPT model, the token embedding matrix and the language modeling head use the same stored weights. This is called **weight tying**. PyTorch stores the shared matrix with dimensions `(vocab_size, hidden_size)` and uses it in the appropriate orientation for the linear calculation.

In [58]:
embedding_weights = model.model.decoder.embed_tokens.weight
lm_head_weights = model.lm_head.weight
print('Embedding weight shape:', tuple(embedding_weights.shape))
print('LM head weight shape:', tuple(lm_head_weights.shape))
print('Same underlying stored weights:', embedding_weights.data_ptr() == lm_head_weights.data_ptr())

Embedding weight shape: (50272, 768)
LM head weight shape: (50272, 768)
Same underlying stored weights: True


### Sampling the Next Token for Generation

The model calculated vocabulary logits for all nine input positions. To choose the next token, we use the logits from the token at the last position in the sequene. The cell below shows the five largest logits. These are scores, not probabilities yet, but we can use them all the same for this purpose.

In [59]:
top5 = logit_outputs[0, -1, :].topk(5)
top5

torch.return_types.topk(
values=tensor([7.6367, 7.5195, 6.9727, 6.6914, 5.6172], dtype=torch.float16,
       grad_fn=<TopkBackward0>),
indices=tensor([    8,    38,    10,    53, 38988]))

In [60]:
top5.values[0]

tensor(7.6367, dtype=torch.float16, grad_fn=<SelectBackward0>)

In [61]:
tokenizer.decode(top5.indices[0])

' and'

---

## 6. Iterative Generation and the KV Cache

Now, we'll use the complete model to iteratively generate text. Pay attention to how many input tokens are processed on each run and what the model saves in the KV cache.

On the first run, the model processes the entire prompt and saves the calculated keys and values for those token positions. This first calculation is sometimes called **prefill**. On later runs, the model processes the new token and reuses the keys and values already saved for the earlier tokens. This is the **decoding** stage.

The cache contains K and V activations calculated for the input. It does not contain the learned K and V projection matrices. A new query still has to be calculated in every decoder block for each new token.

### Reset the Model Inputs

Run this cell before you begin. If you want to start the generation over later, run it again before continuing.

In [ ]:
# Define the input text
input_text = "Hello, I am a language model,"

# Tokenize the input text
inputs = tokenizer(input_text, return_tensors="pt")

# Parse out our "inputs" into individual elements (useful for next cell)
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
past_key_values = None

### Generate One Token at a Time

Run this cell iteratively. To do this, you can click the arrow next to the cell or click Cmd+Enter (Ctrl+Enter on Windows).

Run the next cell at least twice, making sure to inspect the outputs of the cell as they change. The first run uses all of the prompt tokens. Each later run supplies only the token selected on the previous run, along with the saved KV cache.

**Before you run:** Which tensors should grow in sequence length as tokens are generated? Which values are learned model parameters that remain fixed?

In [ ]:
print("New input_ids supplied to this forward pass:", tuple(input_ids.shape))
model_outputs = model(input_ids=input_ids,
                      attention_mask=attention_mask,
                      past_key_values=past_key_values,
                      use_cache=True)

# 1. Get the new token ID (Greedy decoding via argmax)
new_ind = model_outputs.logits[0, -1, :].argmax()

# 2. Decode and update the text
new_word = tokenizer.decode(new_ind)
input_text += new_word
print(input_text)

# 3. Prepare input_ids for the NEXT run (replacing the current inputs)
input_ids = new_ind.unsqueeze(0).unsqueeze(0)

# 4. Prepare past_key_values for the NEXT run
past_key_values = model_outputs.past_key_values

# 5. Prepare attention_mask for the NEXT run
# Append a single '1' to the mask to account for the new token
new_mask_token = torch.ones((1, 1), dtype=torch.long, device=attention_mask.device)
attention_mask = torch.cat([attention_mask, new_mask_token], dim=-1)

# # 6. Looking at past key values. Uncomment when ready for next section.
# print(f"Number of layers in for past_key_values: {len(past_key_values.layers)}")
# print(f"Shape of past keys for layer 0: {past_key_values.layers[0].keys.shape}")
# print(f"Shape of past values for layer 0: {past_key_values.layers[0].values.shape}")

### Look at What the KV Cache Stores

After you have a good sense of the previous exploration, reset the generation. Then, uncomment the last 3 lines of the previous cell.

This output prints the number of decoder blocks represented in the cache and the dimensions of the cached K and V activations for block 0.

Make sure you observe which dimension grows by one after each newly generated token to better understand the content of the KV-cache.

---



# Congratulations!

Great! You're all done. Make sure you answer all the questions on Brightspace and then submit online.

**Optional challenge:** Reset the generation and turn off the cache. Change the loop so that it supplies the complete token history on every run. You may also replace greedy selection with temperature sampling. These changes are optional and are not part of the Brightspace submission.